In [1]:
# importer
import pandas as pd

'''
TF-IDF: Term Frequency-Inverse Document Frequence
Omvandlar text till numeriska vektorer baserat på hur "viktiga" orden är i varje film jämfört hela datasettet
'''
from sklearn.feature_extraction.text import TfidfVectorizer

'''
Cosine simularity matrix
Används för att mäta hur lika två filmer är genom att jämföra dessa TF-IDF vektorer
'''
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("../dataset/movies_cleaned.csv")
df.head()

,title,genres
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy
2,Grumpier Old Men (1995),Comedy|Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance
4,Father of the Bride Part II (1995),Comedy


In [3]:
'''
Skapar en ny kolumn vid namn "features" och tar bort avdelare | och ersätter med mellanrum
'''
df["features"] = df["genres"].str.replace("|", " ", regex=False)

In [4]:
df.head()

,title,genres,features
0,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Adventure Animation Children Comedy Fantasy
1,Jumanji (1995),Adventure|Children|Fantasy,Adventure Children Fantasy
2,Grumpier Old Men (1995),Comedy|Romance,Comedy Romance
3,Waiting to Exhale (1995),Comedy|Drama|Romance,Comedy Drama Romance
4,Father of the Bride Part II (1995),Comedy,Comedy


In [5]:
# TF-IDF
'''
tfidf: skapar en TF-IDF-vektoriserare men som ignorerar "vanliga" engelska ord (and, if, the, ...)
tfidf_matrix: omvandlar filmgenres till numeriska vektorer i matriser. varje rad representerar en film
'''
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["features"])

In [6]:
# Cosine simularity
'''
Jämför samtliga filmer och skapar en "similarity"-matris i vilken högre värden betyder att filmer är mer lika
'''
similarity = cosine_similarity(tfidf_matrix)

# Recommend movie function

In [31]:
def recommend_movies(movie_title):
    '''
    Tar in en titel, och om den finns i datasettet, så matchas dess features mot andra filmer
    och ger tillbaka en lista med de 5 mest liknande
    '''

    if movie_title not in df["title"].values:
        return "Movie not found"
    
    idx = df[df["title"] == movie_title].index[0] # hämtar indexet för den film användaren angav (om filmen finns)

    # hämtar alla similarity scores för angiven film
    # och skapar en lista som med enumerate blir: (film_index, similarity score)
    # ex: [(0, 1.0), (1, 0,5), (2, 0.6), ...]
    sim_scores = list(enumerate(similarity[idx])) 

    # x: x[1] -> similiarity score (x[0] är indexet som det ligger på)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True) # sorterar dessa i fallande ordning efter similarity score

    # tar bort input-filmen från listan, så att den inte kan rekommendera sig själv
    filtered_scores = []
    for score in sim_scores:
        if score[0] != idx:
            filtered_scores.append(score)
    sim_scores = filtered_scores

    sim_scores = sim_scores[:5] # fem första

    # gammal kod
    # sim_scores = sim_scores[1:6] # hoppar över första filmen (som användaren angav) och ger tillbaka de nästkommande 5 högst rankade

    # lägger indexet för ovan filmer i en egen lista
    movie_indices = []
    for i in sim_scores:
        movie_indices.append(i[0])

    return df["title"].iloc[movie_indices] # använder ovan index för att med iloc lokalisera och returnera titlarna

# test av ovan funktion
print(recommend_movies("Antz (1998)"))

# obs: filmnamnet måste vara exakt samma som angivet i movies_cleaned

0                                     Toy Story (1995)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: str


In [32]:
# test av ovan funktion
print(recommend_movies("Jumanji (1995)"))

53             Indian in the Cupboard, The (1995)
109             NeverEnding Story III, The (1994)
767               Escape to Witch Mountain (1975)
1514    Darby O'Gill and the Little People (1959)
1556                          Return to Oz (1985)
Name: title, dtype: str


In [34]:
# test av ovan funktion
print(recommend_movies("Antz (1998)"))

0                                     Toy Story (1995)
2355                                Toy Story 2 (1999)
2809    Adventures of Rocky and Bullwinkle, The (2000)
3000                  Emperor's New Groove, The (2000)
3568                             Monsters, Inc. (2001)
Name: title, dtype: str
